# finemapping

In [208]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import tabulate as tabul
import csv
import statistics
import plotly.express as px

In [209]:
def indexToPosition(index):
    return "" + chr(int(index / 24) + 65) + str(index % 24 + 1)

In [210]:
def positionToIndex(position):
    letter = (ord(position[0]) - 65) * 24
    number = int(position[1:]) - 1

    return letter + number

In [211]:
def loadSynLibrary():
    aminoSeq = "MNYLRRRLSDSNFMANLPNGYMTDLQRPQPPPPPPGAHSPGATPGPGTATAERSSGVAPAASPAAPSPGSSGGGGFFSSLSNAVKQTTAAAAATFSEQVGGGSGGAGRGGAASRVLLVIDEPHTDWAKYFKGKKIHGEIDIKVEQAEFSDLNLVAHANGGFSVDMEVLRNGVKVVRSLKPDFVLIRQHAFSMARNGDYRSLVIGLQYAGIPSVNSLHSVYNFCDKPWVFAQMVRLHKKLGTEEFPLIDQTFYPNHKEMLSSTTYPVVVKMGHAHSGMGKVKVDNQHDFQDIASVVALTKTYATAEPFIDAKYDVRVQKIGQNYKAYMRTSVSGNWKTNTGSAMLEQIAMSDRYKLWVDTCSEIFGGLDICAVEALHGKDGRDHIIEVVGSSMPLIGDHQDEDKQLIVELVVNKMAQALPRQRQRDASPGRGSHGQTPSPGALPLGRQTSQQPAGPPAQQRPPPQGGPPQPGPGPQRQGPPLQQRPPPQGQQHLSGLGPPAGSPLPQRLPSPTSAPQQPASQAAPPTQGQGRQSRPVAGGPGAPPAARPPASPSPQRQAGPPQATRQTSVSGPAPPKASGAPPGGQQRQGPPQKPPGPAGPTRQASQAGPVPRTGPPTTQQPRPSGPGPAGRPKPQLAQKPSQDVPPPATAAAGGPPHPQLNKSQSLTNAFNLPEPAPPRPSLSQDEVKAETIRSLRKSFASLFSD"
    domains = []
    domainDict = {
        "domainName": "A",
        "aminoSequence": aminoSeq[0:28],
        "startIndex": 0,
        "endIndex": 27,
        "startSpot": 6,
        "endSpot": 15,
    }
    domains.append(domainDict.copy())
    domainDict = {
        "domainName": "B",
        "aminoSequence": aminoSeq[28:112],
        "startIndex": 28,
        "endIndex": 111,
        "startSpot": 16,
        "endSpot": 43,
    }
    domains.append(domainDict.copy())
    domainDict = {
        "domainName": "C",
        "aminoSequence": aminoSeq[112:420],
        "startIndex": 112,
        "endIndex": 419,
        "startSpot": 44,
        "endSpot": 145,
    }
    domains.append(domainDict.copy())
    domainDict = {
        "domainName": "D",
        "aminoSequence": aminoSeq[420:655],
        "startIndex": 420,
        "endIndex": 654,
        "startSpot": 146,
        "endSpot": 224,
    }
    domains.append(domainDict.copy())
    domainDict = {
        "domainName": "E",
        "aminoSequence": aminoSeq[655:705],
        "startIndex": 655,
        "endIndex": 704,
        "startSpot": 225,
        "endSpot": 235,
    }
    domains.append(domainDict.copy())
    return domains

In [212]:
def hasLowVariance(dataSet):
    stringency = 0.1

    avg = 0
    maximum = 0
    count = 0
    for spot in dataSet:
        count += 1
        avg += float(spot["rawIntensity"])
        if float(spot["rawIntensity"]) > maximum:
            maximum = float(spot["rawIntensity"])

    avg /= count
    if avg / maximum <= stringency:
        return False

    return True

In [213]:
def checkLowVariance(spots):
    if len(spots) <= 0:
        return True
    stringency = 0.25

    avg = sum(spots) / len(spots)
    maximum = max(spots)

    if avg / maximum <= stringency:
        return False

    return True

In [238]:
class Epitope:
    def __init__(self, startFlank, endFlank, patients):
        self.startFlank = startFlank
        self.endFlank = endFlank
        self.length = self.endFlank - self.startFlank + 1
        self.patients = patients
        self.subEpitopes = []
        self.subPatients = []
        self.superEpitopes = []

    def getStartFlank(self):
        return self.startFlank

    def getEndFlank(self):
        return self.endFlank

    def getPatients(self):
        return self.patients

    def getLength(self):
        return self.length

    def addPatients(self, newPatients):
        for pat in newPatients:
            if pat not in self.patients:
                self.patients.append(pat)

    def addSubEpitopes(self, newSubEpitopes):
        for sEpi in newSubEpitopes:
            if sEpi not in self.subEpitopes:
                self.subEpitopes.append(sEpi)

    def addSuperEpitopes(self, newSuperEpitope):
        for sEpi in newSuperEpitope:
            if sEpi not in self.subEpitopes:
                self.superEpitopes.append(sEpi)

    def addSubPatients(self, newSubPatient):
        for sPat in newSubPatient:
            if sPat in self.subPatients:
                self.subPatients.append(sEpi)

    def getSubPatients(self):
        return self.subPatients

    def getSubEpitopes(self):
        return self.subEpitopes

    def getSuperEpitopes(self):
        return self.superEpitopes

    def isEpiIdentical(self, epitope):
        if (
            self.startFlank == epitope.getStartFlank()
            and self.endFlank == epitope.getEndFlank()
        ):
            patients = epitope.getPatients()
            self.addPatients(patients)
            for pat in patients:
                pat.updateEpiReference(epitope, self)

            self.addSubEpitopes(epitope.getSubEpitopes())
            self.addSuperEpitopes(epitope.getSuperEpitopes())

            # print(indexToPosition(self.startFlank) + " : "+ indexToPosition(epitope.getStartFlank())
            # +" --> "+ indexToPosition(self.endFlank) + " : " + indexToPosition(epitope.getEndFlank()))
            return True
        return False

    def hasEpi(self, epitope):
        if (
            self.startFlank <= epitope.getStartFlank()
            and self.endFlank >= epitope.getEndFlank()
        ):
            self.addSubPatients(epitope.getPatients())
            self.subEpitopes.append(epitope)
            epitope.addSuperEpitopes([self])
            return True
        return False

    def fuseWith(self, epitope):
        if self.startFlank > epitope.getStartFlank():
            self.startFlank = epitope.getStartFlank()

        if self.endFlank < epitope.getEndFlank():
            self.endFlank = epitope.getEndFlank()

        self.addPatients(epitope.getPatients())
        self.addSubPatients(epitope.getSubPatients())
        self.addSubEpitopes(epitope.getSubEpitopes())
        self.addSuperEpitopes(epitope.getSuperEpitopes())

        del epitope

    def getOverlapLength(self, epitope):
        if (
            self.startFlank > epitope.getEndFlank()
            or self.endFlank < epitope.getStartFlank()
        ):
            return 0

        if (
            self.startFlank <= epitope.getStartFlank()
            and self.endFlank <= epitope.getEndFlank()
        ):
            return epitope.getEndFlag() - self.startFlag

        if (
            self.startFlank >= epitope.getStartFlank()
            and self.endFlank >= epitope.getEndFlank()
        ):
            return self.endFlag - epitope.getStartFlag()

        if (
            self.startFlank >= epitope.getStartFlank()
            and self.endFlank <= epitope.getEndFlank()
        ):
            return self.endFlag - self.startFlag

        if self.length <= epitope.getLength():
            return self.lenght
        else:
            return epitope.getLength()

    def getPatientNames(self, patient):
        ret = ""
        for pat in patient:
            ret += pat.getName() + "; "

        return ret[:-2]

    def toString(self):
        return (
            indexToPosition(self.startFlank)
            + " - "
            + indexToPosition(self.endFlank)
            + " ("
            + str(self.length)
            + ")"
        )

    def showContent(self, indent, minLen):
        if minLen <= self.length:
            print(
                indent
                + indexToPosition(self.startFlank)
                + " - "
                + indexToPosition(self.endFlank)
                + " -> "
                + str(self.length)
            )
            print(
                indent
                + str(len(self.patients))
                + " sets: "
                + self.getPatientNames(self.patients)
            )
            if len(self.subEpitopes) > 0:
                print(indent + str(len(self.subEpitopes)) + " sub sets: ")
            indent += "> "
            for subEp in self.subEpitopes:
                print(indent + "++++++")
                subEp.showContent(indent, 0)
            #             print("> "+ pat.getName())
            #             pat.listEpis()
            if indent == "> ":
                print("----------------------------------------------------")

    def getRecursiveTable(self):
        ret = []

        domains = getDomainList(int(self.startFlank), int(self.endFlank))
        if not domains:
            domains = "none"

        resArea = getResiduePosition(int(self.startFlank), int(self.endFlank))

        content = (
            indexToPosition(self.startFlank)
            + " - "
            + indexToPosition(self.endFlank)
            + " (length: "
            + str(self.length)
            + "; domain: "
            + domains
            + ") residues: "
            + resArea
            + " -> "
            + self.getPatientNames(self.patients)
        )
        ret.append(content)

        subRet = []
        for sEpi in self.subEpitopes:
            subRet.append(sEpi.getRecursiveTable())
        if subRet:
            ret.append(subRet)

        return ret

    def getIcicleTable(self):
        ret = []

        domains = getDomainList(int(self.startFlank), int(self.endFlank))
        if not domains:
            domains = "none"

        resArea = getResiduePosition(int(self.startFlank), int(self.endFlank))

        content = (
            indexToPosition(self.startFlank)
            + " - "
            + indexToPosition(self.endFlank)
            + " (length: "
            + str(self.length)
            + "; domain: "
            + domains
            + ") residues: "
            + resArea
            + " -> "
            + self.getPatientNames(self.patients)
        )
        ret.append([content, self.length])

        subRet = []
        for sEpi in self.subEpitopes:
            subRet.append(sEpi.getIcicleTable())
        if subRet:
            ret.append(subRet)

        return ret

In [239]:
def getResiduePosition(startPos, endPos):
    peptideLength = 20
    aminoShift = 3

    spotInset = 6

    startPos -= spotInset
    endPos -= spotInset

    return (
        str((startPos * aminoShift) + 1)
        + " - "
        + str(endPos * aminoShift + peptideLength)
    )

In [240]:
def getDomainList(startFlank, endFlank):
    syn = loadSynLibrary()
    ret = ""
    for domain in syn:
        if (domain["startSpot"] <= startFlank and domain["endSpot"] >= startFlank) or (
            domain["startSpot"] <= endFlank and domain["endSpot"] >= endFlank
        ):
            ret += domain["domainName"] + "; "

    return ret[0:-2]

In [241]:
class Patient:
    def __init__(self, name, type, epitopes):
        self.name = name
        self.type = type
        self.epitopes = epitopes

    def showContent(self):
        print(self.name)
        for epi in self.epitopes:
            print("> " + epi.toString())
        print("------------------")

    def listEpis(self):
        for epis in self.epitopes:
            print(
                ">> "
                + indexToPosition(epis.getStartFlank())
                + " - "
                + indexToPosition(epis.getEndFlank())
            )

    def addEpitopes(self, epis):
        for epi in epis:
            if not epi in self.epitopes:
                self.epitopes.append(epi)

    def getName(self):
        return self.name

    def getEpitopes(self):
        return self.epitopes

    # This method is to be used before deleting duplicate epitopes
    def updateEpiReference(self, oldEpi, newEpi):
        for counter, ep in enumerate(self.epitopes):
            if ep == oldEpi:
                self.epitopes[counter] = newEpi
                break

In [242]:
def splitSetsIntoTiers(nTiers, maxValue, data):
    tData = []
    tiers = ["<= 25%", "<= 50%", "<= 75%", "<= 100%"]
    for i in range(nTiers):
        tData.append([])

    tierStep = maxValue / nTiers

    for counter, dataSet in enumerate(data):
        maxPoint = 0
        for dataPoint in dataSet["data"]:
            if float(dataPoint["rawIntensity"]) > maxPoint:
                maxPoint = float(dataPoint["rawIntensity"])

        for i in range(nTiers):
            if maxPoint <= tierStep * (i + 1):
                # dataSet["name"]+= " ("+ tiers[i] +") "
                tData[i].append(dataSet)
                break

    return tData

In [243]:
def scanFile(data):
    types = ["positive", "negative", "both"]
    tiers = ["<= 25%", "<= 50%", "<= 75%", "<= 100%"]
    typeIndex = 0

    spotNames = []
    rawData = []
    normalizedData = []
    positives = []
    negatives = []

    for sample in data["samples"]:
        if not hasLowVariance(sample["data"]):
            if sample["type"][0] == types[0]:
                positives.append(sample)
            else:
                negatives.append(sample)

    positives = splitSetsIntoTiers(4, (2**16 - 1), positives)
    negatives = splitSetsIntoTiers(4, (2**16 - 1), negatives)

    epiAnalysis(positives[1] + positives[2] + positives[3], "positives")
    epiAnalysis(negatives[1] + negatives[2] + negatives[3], "negatives")

In [244]:
def epiProfile(sample, minLength):
    inset = 6  # Starting position of actual sample
    stringency = getSignalThreshold(sample, inset)
    print(sample["name"])
    print("stringency: " + str(stringency))
    spotList = []

    # Binary map hit/miss for all spots
    for counter, spot in enumerate(sample["data"]):
        if float(spot["normalizedIntensity"]) >= stringency:
            spotList.append(True)
            print(indexToPosition(counter))
        else:
            spotList.append(False)

    print("---------------")
    # Bridges "accidental" one spot gaps
    for counter, spot in enumerate(spotList):
        if counter != 0 and counter < len(
            spotList
        ):  # do not check on first or last spot
            if not spot and spotList[counter - 1] and spotList[counter + 1]:
                spotList[counter] = True

    # Scan for continuety and store flanks as an instance of Epitope
    startIndex = -1
    epis = []

    pat = Patient(sample["name"], sample["type"], epis)  # patient has epitopes

    for counter, spot in enumerate(spotList):
        if spot and startIndex < 0:
            startIndex = counter
        elif not spot and startIndex >= 0:
            if (counter - startIndex) >= minLength:
                epis.append(
                    Epitope(startIndex, (counter - 1), [pat])
                )  # epitope has patients -> bidirectional association
            startIndex = -1

    pat.addEpitopes(epis)
    return pat

In [245]:
def epiProfiler(sample, minLength):
    inset = 6  # Starting position of actual sample
    spots = []
    for spot in sample["data"][inset:]:
        spots.append(float(spot["normalizedIntensity"]))

    stringency = getSignalThresholdsExp(spots)
    stringency.insert(0, 1.1)
    print(sample["name"])

    spotList = []
    print(stringency)
    stringency = stringency[-1]
    # Binary map hit/miss for all spots
    for counter, spot in enumerate(sample["data"]):
        if float(spot["normalizedIntensity"]) >= stringency:
            spotList.append(True)

        else:
            spotList.append(False)

    # Scan for continuety and store flanks as an instance of Epitope
    startIndex = -1
    epis = []

    pat = Patient(sample["name"], sample["type"], epis)  # patient has epitopes

    for counter, spot in enumerate(spotList):
        if spot and startIndex < 0:
            startIndex = counter
        elif not spot and startIndex >= 0:
            if (counter - startIndex) >= minLength:
                epis.append(
                    Epitope(startIndex, (counter - 1), [pat])
                )  # epitope has patients -> bidirectional association
            startIndex = -1

    pat.addEpitopes(epis)
    return pat

In [246]:
def getSignalThreshold(sample, startPos=0):
    spots = []
    for spot in sample["data"][startPos:]:
        spots.append(float(spot["normalizedIntensity"]))

    spots.sort()
    spotsDelta = []

    for counter, spot in enumerate(spots[:-2]):
        spotsDelta.append(spots[counter + 1] - spot)

    maxDeltaIndex = spotsDelta.index(max(spotsDelta))

    return spots[maxDeltaIndex + 1]

In [247]:
def getSignalThresholds(spots):

    spots.sort()
    spotsDelta = []
    ret = []
    for counter, spot in enumerate(spots[:-2]):
        spotsDelta.append(spots[counter + 1] - spot)
    if len(spotsDelta) < 1:
        return ret

    maxDeltaIndex = spotsDelta.index(max(spotsDelta))

    ret += [spots[maxDeltaIndex + 1]]
    if not checkLowVariance(spots[: maxDeltaIndex + 1]):
        ret += getSignalThresholds(spots[: maxDeltaIndex + 1])
    return ret

In [248]:
def getSignalThresholdsExp(spots):

    spots.sort()
    spotsDelta = []
    ret = []
    for counter, spot in enumerate(spots[:-2]):
        spotsDelta.append(spots[counter + 1] - spot)
    if len(spotsDelta) < 1:
        return ret

    relDelta = []
    for counter, sDelta in enumerate(spotsDelta):
        if counter > 0:
            relDelta.append(sDelta - spotsDelta[counter - 1])

    maxDeltaIndex = relDelta.index(max(relDelta))

    ret += [spots[maxDeltaIndex + 1]]
    if not checkLowVariance(spots[: maxDeltaIndex + 1]):
        ret += getSignalThresholds(spots[: maxDeltaIndex + 1])
    return ret

In [249]:
def getSignalThresholdsNew(spots):

    spots.sort()
    spotsDelta = []
    ret = []
    for counter, spot in enumerate(spots[:-2]):
        spotsDelta.append(spots[counter + 1] - spot)

    maxDeltaIndex = spotsDelta.index(max(spotsDelta))

    ret += [spots[maxDeltaIndex + 1]]
    if getAvgMaxDelta(spots[: maxDeltaIndex + 1]) > max(spotsDelta):
        ret += getSignalThresholds(spots[: maxDeltaIndex + 1])
    return ret

In [250]:
def getAvgMaxDelta(spots):
    return max(spots) - (sum(spots) / len(spots))

In [251]:
def mergeDuplicates(epiSet):
    delIndices = []
    for counter, epis in enumerate(epiSet):
        for secCounter, runEpis in enumerate(epiSet[(counter + 1) :]):
            if not (counter + secCounter + 1) in delIndices:
                if epis.isEpiIdentical(runEpis):
                    delIndices.append(counter + (secCounter + 1))

    delIndices.sort(reverse=True)
    for index in delIndices:
        del epiSet[index]

    return epiSet

In [252]:
def subSetEpis(epiSet):
    mergeDuplicates(epiSet)
    isSubset = False
    subsetIndices = []
    for counter, epis in enumerate(epiSet):
        for secCounter, runEpis in enumerate(epiSet):
            if not counter == secCounter:
                if runEpis.hasEpi(epis):
                    subsetIndices.append(counter)

    subsetIndices = list(dict.fromkeys(subsetIndices))
    for index in reversed(subsetIndices):
        del epiSet[index]

    trimEpiTree(epiSet)  # removing elements that exist on a lower level

In [253]:
def trimEpiTree(epiSet):
    subEpis = []
    for epi in epiSet:
        subEpis += epi.getSubEpitopes()

    delIndices = []
    for counter, epi in enumerate(epiSet):
        if epi in subEpis:
            delIndices.append(counter)

    for index in reversed(delIndices):
        del epiSet[index]

    for epi in epiSet:
        subEpitopes = epi.getSubEpitopes()
        if subEpitopes:
            trimEpiTree(subEpitopes)

In [254]:
def getPrintoutString(items, text, level=0):
    subitems = []
    for item in items:
        if len(item) > 1:
            content, subitems = item
        else:
            content = item[0]
            subitems = []
        text.append(str(level) + content)
        if subitems:
            getPrintoutString(subitems, text, level + 1)

In [255]:
def print_stuff(items, level=0):
    subitems = []
    for item in items:
        if len(item) > 1:
            content, subitems = item
        else:
            content = item[0]
        print("\t" * level + content)
        if subitems:
            print_stuff(subitems, level + 1)
        if level == 0:
            print("------------------------------------------------------------------")

In [256]:
def epiAnalysis(sampleSet, title):
    print(len(sampleSet))
    patProfiles = []
    for pat in sampleSet:
        patProfiles.append(epiProfiler(pat, 1))
    for pat in patProfiles:
        pat.showContent()

    print("-------------------------")
    epiProfiles = []
    for pat in patProfiles:
        epiProfiles += pat.getEpitopes()

    subSetEpis(epiProfiles)

    epiProfiles.sort(key=lambda x: x.startFlank, reverse=False)

    printEpiTable(epiProfiles, title)


#     for epis in epiProfiles:
#         epis.showContent("",0)

In [257]:
def printEpiTable(epiProfiles, title):
    recursiveTable = []
    for epiProfile in epiProfiles:
        recursiveTable.append(epiProfile.getRecursiveTable())

    text = []
    getPrintoutString(recursiveTable, text)
    maxDepth = 0
    for t in text:
        if int(t[0]) > maxDepth:
            maxDepth = int(t[0])
    maxDepth += 1

    tab = [""] * maxDepth

    table = []
    for counter, t in enumerate(text):
        tab[int(t[0])] = t[1:]

        if counter + 1 < len(text):
            if int(text[counter + 1][0]) <= int(t[0]):
                table.append(tab.copy())
                tab = [""] * maxDepth
    #                 if(int(text[counter+1][0]) == 0):
    #                     table.append(["-"*80]*maxDepth)
    table.append(tab.copy())

    file = open(
        "C:/Users/KaiKr/Desktop/plots/SYNI-epitopesNew - " + title + ".csv", "w"
    )
    csvWriter = csv.writer(file)

    for tabu in table:
        text = ""
        for t in tabu:
            text += t + ","
        print(text)
        csvWriter.writerow(text.split(","))
    file.close()

    print(tabul.tabulate(table, tablefmt="github"))
    icicleChart(epiProfiles)

In [266]:
def icicleChart(epiProfiles):
    epitope = []
    parent = []
    value = []
    for count, eProfile in enumerate(epiProfiles):
        iceTable = eProfile.getIcicleTable()
        for depth, eP in enumerate(iceTable):
            print(eP)
            epitope.append(eP[0])
            if depth > 0:
                parent.append(iceTable[depth - 1][0])
            else:
                parent.append("")
            value.append(eP[1])

    data = dict(epitope, parent, value)

    fig = px.icicle(
        data,
        names="epitope",
        parents="parent",
        values="value",
    )
    fig.update_traces(root_color="lightgrey")
    fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
    fig.show()

Icicle chart, flame chart

In [267]:
dPath = "C:/Users/KaiKr/Desktop/plots/SYNI.json"
with open(dPath) as file:
    data = json.load(file)
scanFile(data)

32
SM81
[1.1, 0.3226966451334478]
SM17
[1.1, 0.4405145408249281, 0.33743774279242356]
SM19
[1.1, 0.43626170542616205, 0.24332542374422794]
SM52
[1.1, 0.42882976850002663, 0.2199523719572775]
SMIII
[1.1, 0.2603187579858277]
SC1
[1.1, 0.2048134987606369]
SC15
[1.1, 0.2221675586257183, 0.10986347668810824]
SC77
[1.1, 0.5507833105805658, 0.17250974732552216]
SM78
[1.1, 0.39823648154719166, 0.2752757278542632, 0.21626180202476766]
SC3
[1.1, 0.6541404947051539, 0.567928908787822, 0.42507390149325713, 0.2738744977014351]
M017
[1.1, 0.2746071456862533, 0.2305170931372223]
M026
[1.1, 0.3754947197085236, 0.2301082542434306]
M058
[1.1, 0.6605562668024043, 0.3208197216300795]
M095
[1.1, 0.18814124279978622]
SM5
[1.1, 0.4131629975977728, 0.3131045868320941, 0.19136000534818476]
SM13
[1.1, 0.17269229472351957, 0.1286642250352145]
MX109
[1.1, 0.19591422051775842]
MX138
[1.1, 0.2647712622679553, 0.18018488718253436]
MX140
[1.1, 0.3018494853834127, 0.24742126606353093, 0.16574263984346485]
SV04
[1.1, 0

IndexError: list index out of range